# 举例1：大模型分析工具的使用

In [1]:
from langchain_classic.chains.question_answering.map_reduce_prompt import messages
#1.导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 2.定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3.获取工具的列表
tools = [MoveFileTool]

# 4.因为大模型invoke调用时，需要传入函数的列表，所以需要将工具转换为函数：convert_to_openai_function()
functions = [convert_to_openai_function(t) for t in tools ]

#5.获取消息列表
messages = [HumanMessage(content="将文件a移动到桌面")]

# 6.调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools=tools, 不支持
    functions=functions,
)

print(response)

D:\conda_envs\python313\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
D:\conda_envs\python313\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value <class 'langchain_community.tools.file_management.move.FileMoveInput'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


content='' additional_kwargs={'function_call': {'arguments': '{"args_schema":{"source":"a","destination":"桌面/a"}}', 'name': 'MoveFileTool'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 202, 'total_tokens': 227, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DKIxbBwn7ckc97WJuxc9KFqk0I4VU', 'finish_reason': 'function_call', 'logprobs': None} id='lc_run--019cfaa7-90d1-7761-8856-46cd5efd6c8e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 202, 'output_tokens': 25, 'total_tokens': 227, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [2]:
#5.获取消息列表
messages = [HumanMessage(content="查询一下北京的天气")]

# 6.调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools=tools, 不支持
    functions=functions,
)

print(response)

content='抱歉，我无法提供实时天气信息。建议您使用天气应用或网站查询北京的最新天气情况。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 200, 'total_tokens': 225, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DKIxwb8lLhy7mB6Hqb57IW77qvqpU', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cfaa7-e612-7a71-9ec8-4dde2cb171a4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 200, 'output_tokens': 25, 'total_tokens': 225, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


通过上面两个测试发现，得到的AIMessage的核心属性如下：
1、如果分析出需要调用对应的工具：
content:信息为空。因为大模型要调用工具，所以就不会直接返回信息给用户

additional_kwargs ：包含function_call字段，指明具体函数调用的参数和函数名



2、如果分析出不需要调用对应的工具
content:信息不为空

additional_kwargs：不包含function_call字段


# 举例2：如何调用具体大模型分析出来的工具

说明：

1.大模型与Agent的核心区别:是否涉及到工具的调用

2.针对大模型：仅能分析出要调用的工具，但是此工具（或函数）不能真正的执行

针对于Agent：除了分析出要调用的工具外，还可以执行具体的工具（或函数）

In [21]:
from langchain_classic.chains.question_answering.map_reduce_prompt import messages
#1.导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 2.定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3.获取工具的列表
tools = [MoveFileTool()]

# 4.因为大模型invoke调用时，需要传入函数的列表，所以需要将工具转换为函数：convert_to_openai_function()
functions = [convert_to_openai_function(t) for t in tools ]

#5.获取消息列表
messages = [HumanMessage(content="将当前目录下的文件a.txt移动到D:\\Desk")]

# 6.调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools=tools, 不支持
    functions=functions,
)

print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"source_path":"a.txt","destination_path":"D:\\\\Desk\\\\a.txt"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 82, 'total_tokens': 109, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DKJJQQmP9Q9iUFbHNPv2uio6ZdCb1', 'finish_reason': 'function_call', 'logprobs': None} id='lc_run--019cfabc-3821-7331-a2a3-ea6a23c4ddde-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 82, 'output_tokens': 27, 'total_tokens': 109, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [22]:
import json

if "function_call" in response.additional_kwargs:
    tool_name = response.additional_kwargs["function_call"]["name"]
    tool_args = json.loads(response.additional_kwargs["function_call"]["arguments"])
    print(f"调用工具：{tool_name} \n 参数：{tool_args}")
else:
    print(f"模型回复：{response.content}")

调用工具：move_file 
 参数：{'source_path': 'a.txt', 'destination_path': 'D:\\Desk\\a.txt'}


In [23]:
if "move_file" in response.additional_kwargs["function_call"]["name"]:
    tool = MoveFileTool()
    result = tool.run(tool_args)
    print("工具执行的结果",result)

工具执行的结果 File moved successfully from a.txt to D:\Desk\a.txt.
